# Data collection for the year 2020

In [1]:
import cocopp
dsl = cocopp.load("bbob/2020/*")

  downloading https://numbbo.github.io/data-archive/data-archive/bbob/2020/CMA-ES-2019_Hansen.tgz to C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cache\das\da\bbob\2020\CMA-ES-2019_Hansen.tgz
  downloading https://numbbo.github.io/data-archive/data-archive/bbob/2020/HE-ES_Glasmachers.tgz to C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cache\das\da\bbob\2020\HE-ES_Glasmachers.tgz
  downloading https://numbbo.github.io/data-archive/data-archive/bbob/2020/SLSQP+lq-CMA-ES_Hansen.tgz to C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cache\das\da\bbob\2020\SLSQP+lq-CMA-ES_Hansen.tgz
  downloading https://numbbo.github.io/data-archive/data-archive/bbob/2020/SLSQP-11-scipy_Hansen.tgz to C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cache\das\da\bbob\2020\SLSQP-11-scipy_Hansen.tgz
  downloading https://numbbo.github.io/data-archive/data-archive/bbob/2020/lq-CMA-ES_Hansen.tgz to C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cache\das\da\bbob\2020\lq-CMA-ES_Hansen.tgz
    archive extracted to folder C:\Users\e

In [2]:
import numpy as np

dd = dsl.dictByDimFunc()     # your grouped datasets
t = 1e-8                     # choose the target precision

best_by_df = {}              # (dim, fid) -> (best_alg, best_ert)

for dim in sorted(dd.keys()): 
    for fid in sorted(dd[dim].keys()):
        rows = []
        for ds in dd[dim][fid]:                 # each ds = one algorithm
            ert = float(ds.detERT([t])[0])      # ERT in #evals at target t
            rows.append((ds.algId, ert))  
        # ignore INF (not reached) when picking best
        finite = [(a, e) for (a, e) in rows if np.isfinite(e)] 
    
        if finite:
            best_alg, best_ert = min(finite, key=lambda x: x[1]) 
        else:
            best_alg, best_ert = None, np.inf
        best_by_df[(dim, fid)] = (best_alg, best_ert) 
        print(f"dim={dim:>2}, F{fid:>2} -> {best_alg}  (ERT={best_ert:.3g} @ {t})")


dim= 2, F 1 -> SLSQP+lq-CMA-ES_Hansen  (ERT=7 @ 1e-08)
dim= 2, F 2 -> SLSQP+lq-CMA-ES_Hansen  (ERT=56.2 @ 1e-08)
dim= 2, F 3 -> lq-CMA-ES_Hansen  (ERT=1.86e+03 @ 1e-08)
dim= 2, F 4 -> SLSQP-11-scipy_Hansen  (ERT=4.74e+03 @ 1e-08)
dim= 2, F 5 -> lq-CMA-ES_Hansen  (ERT=11.9 @ 1e-08)
dim= 2, F 6 -> SLSQP-11-scipy_Hansen  (ERT=293 @ 1e-08)
dim= 2, F 7 -> lq-CMA-ES_Hansen  (ERT=219 @ 1e-08)
dim= 2, F 8 -> SLSQP-11-scipy_Hansen  (ERT=121 @ 1e-08)
dim= 2, F 9 -> SLSQP+lq-CMA-ES_Hansen  (ERT=73 @ 1e-08)
dim= 2, F10 -> lq-CMA-ES_Hansen  (ERT=147 @ 1e-08)
dim= 2, F11 -> lq-CMA-ES_Hansen  (ERT=133 @ 1e-08)
dim= 2, F12 -> SLSQP+lq-CMA-ES_Hansen  (ERT=261 @ 1e-08)
dim= 2, F13 -> lq-CMA-ES_Hansen  (ERT=304 @ 1e-08)
dim= 2, F14 -> lq-CMA-ES_Hansen  (ERT=204 @ 1e-08)
dim= 2, F15 -> lq-CMA-ES_Hansen  (ERT=1.95e+03 @ 1e-08)
dim= 2, F16 -> CMA-ES-2019_Hansen  (ERT=659 @ 1e-08)
dim= 2, F17 -> CMA-ES-2019_Hansen  (ERT=1.74e+03 @ 1e-08)
dim= 2, F18 -> CMA-ES-2019_Hansen  (ERT=2.83e+03 @ 1e-08)
dim= 2, F19 -

In [8]:
from collections import Counter, defaultdict

In [9]:
# Build a frequency counter: how many (dim,fid) each algo wins
win_counter = Counter(
    alg for (alg, ert) in best_by_df.values()
    if alg is not None and np.isfinite(ert)
)

# If you want a plain dict:
wins_dict = dict(win_counter)

# (Optional) pretty print, most wins first
for alg, count in win_counter.most_common():
    print(f"{alg}: {count}")

lq-CMA-ES_Hansen: 62
SLSQP+lq-CMA-ES_Hansen: 30
SLSQP-11-scipy_Hansen: 18
CMA-ES-2019_Hansen: 15
HE-ES_Glasmachers: 9


In [10]:
"""
Given best_by_df: {(dim, fid): (alg, ert)},
return {dim: algo_with_most_(fid)_wins_in_that_dim}.
Tie-break: lower total ERT across that dim, then alphabetical.
    """
wins = defaultdict(Counter)                    # dim -> Counter({alg: count})
ert_sums = defaultdict(lambda: defaultdict(float))  # dim -> {alg: total_ert}

for (dim, fid), (alg, ert) in best_by_df.items():
    if alg is None or not np.isfinite(ert):
        continue
    wins[dim][alg] += 1
    ert_sums[dim][alg] += float(ert)

result = {}
for dim, counter in wins.items():
    max_wins = max(counter.values())
    candidates = [a for a, c in counter.items() if c == max_wins]
    best = min(candidates, key=lambda a: (ert_sums[dim][a], a))  # tie-breaks
    result[dim] = best
result


{2: 'lq-CMA-ES_Hansen',
 3: 'lq-CMA-ES_Hansen',
 5: 'lq-CMA-ES_Hansen',
 10: 'lq-CMA-ES_Hansen',
 20: 'lq-CMA-ES_Hansen',
 40: 'lq-CMA-ES_Hansen'}

In [11]:
import numpy as np
import pandas as pd

# Make sure 'dd' already exists
# (if not, run: dsl = cocopp.load('path/to/your/ppdata'); dd = dsl.dictByDimFunc())

targets = [1e-1, 1e-2, 1e-3, 1e-5, 1e-8]
rows = []  # reset before starting the full loop

for dim in sorted(dd.keys()):                      # e.g. [2, 3, 5, 10, 20, 40]
    for fid in sorted(dd[dim].keys()):
        for t in targets:
            algo_erts = []
            for ds in dd[dim][fid]:                # each algorithm
                ert = float(ds.detERT([t])[0])
                algo_erts.append((ds.algId, ert))
            
            finite = [(a, e) for (a, e) in algo_erts if np.isfinite(e)]

            if finite:
                best_alg, best_ert = min(finite, key=lambda x: x[1])
            else:
                best_alg, best_ert = None, np.inf

            rows.append({
                "dimension": dim,
                "function_id": fid,
                "target": t,
                "best_algorithm": best_alg,
                "best_ERT": best_ert
            })

# Build DataFrame
df_best = pd.DataFrame(rows)
df_best = df_best.sort_values(by=["dimension", "function_id", "target"]).reset_index(drop=True)

# Confirm dimensions included
print("✅ Unique dimensions in table:", df_best["dimension"].unique())
print(df_best.head(15))


✅ Unique dimensions in table: [ 2  3  5 10 20 40]
    dimension  function_id        target          best_algorithm     best_ERT
0           2            1  1.000000e-08  SLSQP+lq-CMA-ES_Hansen     7.000000
1           2            1  1.000000e-05  SLSQP+lq-CMA-ES_Hansen     7.000000
2           2            1  1.000000e-03  SLSQP+lq-CMA-ES_Hansen     7.000000
3           2            1  1.000000e-02  SLSQP+lq-CMA-ES_Hansen     7.000000
4           2            1  1.000000e-01  SLSQP+lq-CMA-ES_Hansen     7.000000
5           2            2  1.000000e-08  SLSQP+lq-CMA-ES_Hansen    56.200000
6           2            2  1.000000e-05  SLSQP+lq-CMA-ES_Hansen    44.933333
7           2            2  1.000000e-03  SLSQP+lq-CMA-ES_Hansen    38.000000
8           2            2  1.000000e-02  SLSQP+lq-CMA-ES_Hansen    33.866667
9           2            2  1.000000e-01  SLSQP+lq-CMA-ES_Hansen    29.733333
10          2            3  1.000000e-08        lq-CMA-ES_Hansen  1863.533333
11          2 

In [12]:
import os
os.makedirs("results", exist_ok=True)

df_best.to_csv("results/best_algos_2020.csv", index=False)
